# Q4b -- rolling out the signal under a $1,000 loss tolerance

> If you had no historical data, how would you roll out this signal to maximize total
> expected profit in a three-month window, assuming a max loss tolerance of $1,000, a
> 55% win probability per trade (10% edge over a 50% breakeven), and 500 uncorrelated
> trades per day? Would you paper-trade or use your budget up?

**Step 1 of 4 -- the naive simulation.** Constant bet size, even-money trades, no fees.

### The formula everything rests on

Buy a contract at price `p`. Right -> it settles at $1, so you made `1 - p`. Wrong ->
it settles at $0, so you lost `p`. With a true win rate `w`:

```
EV per contract = w*(1 - p) - (1 - w)*p = w - p
```

Everything cancels. **The edge is the win rate minus the price paid.** So a 55% win
rate is not an edge by itself:

| you buy at | with w = 0.55 | edge |
| --- | --- | --- |
| 0.50 | 0.55 - 0.50 | **+5c** |
| 0.80 | 0.55 - 0.80 | **-25c** |

The prompt's phrase *"10% edge over a 50% breakeven"* only makes sense at `p = 0.50`,
so that is the base case here: win $X or lose $X, 55/45.

### Why the calendar does not matter

500 trades/day for 64 days is 32,000 steps of a random walk. With a constant bet size
nothing in the walk depends on the date -- 250/day for 128 days would be the same
simulation. Ruin at trade 274 is ruin whether that is called Tuesday or not. So the
whole three-month window collapses to one number: **32,000 trades**.

In [1]:
import numpy as np
import pandas as pd

from utils import simulate_naive_random_walk, ruin_probability_closed_form

### Test 1 -- does the simulation agree with the closed form?

Gambler's ruin gives the probability of ever touching a floor `k` bets away as
`((1-w)/w) ** k`. This is not the answer to anything -- it is a **unit test**. If the
Monte Carlo does not land near it, the Monte Carlo has a bug.

One caveat: the closed form assumes infinitely many trades, so it is a slight
over-estimate of a 32,000-trade simulation, which has only finitely many chances to
get there.

In [ ]:
BET_SIZES = [10, 50, 100, 200]

rows = [simulate_naive_random_walk(x, n_paths = 5_000, seed = 1) for x in BET_SIZES]
sim  = pd.DataFrame(rows).set_index("bet_size")
sim["closed_form"] = [ruin_probability_closed_form(x) for x in BET_SIZES]

# Monte Carlo standard error on a proportion, so we can judge "close enough"
sim["mc_std_err"] = np.sqrt(sim["prob_ruin"] * (1 - sim["prob_ruin"]) / 5_000)
sim["diff_in_std_errs"] = ((sim["prob_ruin"] - sim["closed_form"]).abs()
                           / sim["mc_std_err"].replace(0, np.nan))

sim[["prob_ruin", "closed_form", "mc_std_err", "diff_in_std_errs"]].round(4)

In [ ]:
# Assert it, so the notebook fails loudly rather than quietly drifting.
ok = (sim["diff_in_std_errs"].fillna(0) < 3).all()
assert ok, "simulation disagrees with the closed form by more than 3 standard errors"
print("PASS -- every bet size is within 3 standard errors of the closed form")

PASS -- every bet size is within 3 standard errors of the closed form


### Test 2 -- a case where the answer is known without any theory

Two sanity checks that do not depend on the gambler's-ruin formula at all:

- **A fair coin has no edge.** At `win_prob = 0.50` the walk has zero drift, so median
  final P&L should sit at roughly zero.
- **A big enough floor is unreachable.** With a $1 bet, the floor is 1,000 losing bets
  away and a 55% win rate drifts upward the whole time, so ruin should never happen.

In [ ]:
fair = simulate_naive_random_walk(100, win_prob = 0.50, n_paths = 2_000, seed = 7)
print(f"fair coin, $100 bet   -> median final P&L {fair['median_final_pnl']:>10,.0f}"
      f"   (expect ~0)")

tiny = simulate_naive_random_walk(1, n_paths = 2_000, seed = 7)
print(f"$1 bet, 55% win rate  -> prob of ruin     {tiny['prob_ruin']:>10.4f}"
      f"   (expect 0)")

assert abs(fair["median_final_pnl"]) < 5_000, "zero-edge walk should not drift"
assert tiny["prob_ruin"] == 0.0, "a $1 bet should never reach -$1,000 here"
print("\nPASS -- both behave as expected")

fair coin, $100 bet   -> median final P&L       -200   (expect ~0)
$1 bet, 55% win rate  -> prob of ruin         0.0000   (expect 0)

PASS -- both behave as expected


### The full result

In [ ]:
sim[["prob_ruin", "closed_form", "median_ruin_trade",
     "median_final_pnl", "median_worst_pnl", "p05_worst_pnl"]].round(2)

,prob_ruin,closed_form,median_ruin_trade,median_final_pnl,median_worst_pnl,p05_worst_pnl
bet_size,,,,,,
10,0.00,0.00,NaN,32040.0,-30.0,-150.0
50,0.02,0.02,191.0,160200.0,-150.0,-750.0
100,0.14,0.13,71.0,320400.0,-300.0,-1500.0
200,0.36,0.37,26.0,640800.0,-600.0,-3000.0


### The P&L path -- watching the wiggle

Two panels, and the split is necessary rather than decorative. With a 10% edge over
32,000 trades the paths finish in the hundreds of thousands, while the floor we care
about is -$1,000. At full scale the floor and $0 are the same line to the eye. The
right panel rescales to the opening stretch, which is where ruin actually happens.

Paths that ever touch the floor are drawn in red, on top of the survivors.

In [ ]:
from utils import simulate_pnl_paths, plot_pnl_paths
import matplotlib.pyplot as plt

for bet in (100, 200):
    paths = simulate_pnl_paths(bet, n_paths = 40, seed = 3)
    plot_pnl_paths(paths, loss_limit = -1_000, zoom = 1_000)
    plt.suptitle(f"${bet} per trade, 55% win rate", fontsize = 12)
    plt.tight_layout()
    plt.show()

**What the picture shows that the table does not.**

Every red path dips below -$1,000 inside the first few hundred trades, then climbs and
finishes in the same cloud as the survivors. By trade 32,000 you cannot tell them
apart -- at $200 per trade, 20 of 40 paths touched the floor and all 40 end up between
roughly $550k and $750k.

**That similarity is the trap.** The simulation lets ruined paths keep trading. In
reality hitting the loss tolerance means you stop, so those red paths do not earn
$600,000 -- they earn -$1,000 and end. The left panel is what the edge would have paid
if you survived to collect it; the right panel is the part that decides whether you do.

It also makes the sizing tension visible. At $200 the paths climb twice as fast, and
half of them never get to enjoy it.

## What step 1 shows

**1. When ruin happens, it happens almost immediately.** Median trade number at ruin is
**191** at a $50 bet, **71** at $100, **26** at $200. Not day 40 -- inside the first
hour of the first day. Over 32,000 trades the edge overwhelms the noise, so the only
real danger is the opening stretch, before any profit has accumulated to absorb a bad
run. That is the whole question.

**2. This model cannot tell you what size to trade, and it is important to say so.**
Expected profit rises with bet size even after ruin is accounted for: at $200 there is
a 36% chance of blowing up, but the surviving 64% earn so much that betting bigger
still wins on average. A model whose answer is always *bet more* has not answered the
question. The constraint has to come from somewhere else -- Kelly sizing says risk
`2w - 1 = 10%` of bankroll, which is $100 here, and that is the size carrying a 14%
chance of ruin. Step 3's dynamic sizing is what allows something near Kelly without
that blowup risk.

**3. The profit numbers are not credible, and that is a limitation of the setup, not a
finding.** Median final P&L is $32,040 at a $10 bet -- mechanically 32,000 trades x 10%
edge x $10. The simulation is claiming $1,000 becomes $32,000 in three months, and
$640,000 at a $200 bet. That assumes 32,000 genuinely independent 55% opportunities
with no market impact and no fees. Worth stating rather than letting the number stand.

### Still to do

- **Step 2 -- real price paths.** Trap to avoid: do *not* hold `w` fixed at 0.55 while
  feeding in real prices. By `EV = w - p` that gives a +19c edge where a market traded
  at 0.36 and -41c where it traded at 0.96, so the simulation would just be measuring
  the average price of whichever market got picked. Fix: set `w = p + 0.05`, a constant
  5c edge everywhere, which recovers the prompt exactly at `p = 0.50`. The point of
  step 2 is then that expected profit is *identical* across markets while risk of ruin
  is wildly different -- losing 96c on a contract bought at 0.96 versus 10c on one
  bought at 0.10. **With a fixed loss budget, where on the price ladder you trade
  matters as much as the size of your edge.**
- **Step 3 -- dynamic sizing**, resized every trade rather than daily. Wrinkle: purely
  proportional sizing can never reach zero, so `P(ruin)` becomes 0 by construction and
  a different failure definition is needed.
- **Step 4 -- fees.** `p(1-p)*0.07` is 1.75c at `p = 0.50`, against a 5c edge. **35% of
  the edge**, gone.
- **Answer the actual question asked:** paper-trade or use the budget? With 500 trades
  a day, the standard error on the win rate is `sqrt(0.55*0.45/500)` = 2.2%, so a
  single day of live trading at minimum size measures the edge nearly as precisely as
  paper trading would -- while earning rather than costing. That argues for trading
  small immediately and scaling with evidence, not for paper trading.